In [ ]:
import requests
import json
from datetime import datetime, timedelta
import random

# Configuration
BASE_URL = "https://nostradamus-ioto.issel.ee.auth.gr/api/v1"
PROJECT_ID = "Your_Project_ID"  # Replace with your project ID


# API Keys (replace with your actual keys)
MASTER_KEY = 'master_key'
WRITE_KEY = 'write_key'
READ_KEY = 'read_key'

In [ ]:
print("🌱 Nostradamus IoTO API - Soil Data Example")

## Step 1: Create Soil Collection

def create_soil_collection(project_id, master_key):
    """Create soil data collection"""
    url = f"{BASE_URL}/projects/{project_id}/collections"
    headers = {"X-API-Key": master_key}

    collection_data = {
        "name": f"sensors",
        "description": "Soil monitoring data",
        "tags": ["soil", "agriculture", "sensors"],
        "collection_schema": {
            "key": "SOIL_001",                    # Identifying key
            "timestamp": "2025-06-17T10:30:00Z",        # Timestamp
            "soil_moisture": 45.2,                      # Soil moisture %
            "ph_level": 6.8,                           # pH level
            "temperature": 18.5,                        # Temperature °C
            "nitrogen": 45.0,                          # Nitrogen ppm
            "phosphorus": 12.5,                        # Phosphorus ppm
            "potassium": 180.0,                        # Potassium ppm
            "battery_level": 87.3                      # Battery %
        }
    }

    response = requests.post(url, json=collection_data, headers=headers)
    if response.status_code == 200:
        print("✅ Collection created successfully!")
        return response.json()
    else:
        print(f"Failed to create collection: {response.text}")
        return None

collection_result = create_soil_collection(PROJECT_ID, MASTER_KEY)
print(collection_result)

In [ ]:
collection_id='Your_collection_id'
def generate_soil_data(num_sensors=3, num_readings=24):
    """Generate soil sensor data"""
    data = []
    base_time = datetime.now() - timedelta(hours=24)

    for sensor_num in range(1, num_sensors + 1):
        sensor_id = f"SOIL_{sensor_num:03d}"

        for hour in range(num_readings):
            timestamp = base_time + timedelta(hours=hour)

            reading = {
                "key": sensor_id,
                "timestamp": timestamp.strftime("%Y-%m-%dT%H:%M:%S"),
                "soil_moisture": round(random.uniform(30, 60), 1),
                "ph_level": round(random.uniform(6.0, 7.5), 1),
                "temperature": round(random.uniform(15, 25), 1),
                "nitrogen": round(random.uniform(30, 60), 1),
                "phosphorus": round(random.uniform(8, 20), 1),
                "potassium": round(random.uniform(120, 220), 1),
                "battery_level": round(random.uniform(70, 100), 1)
            }
            data.append(reading)

    return data

def send_data(project_id, collection_id, write_key, data):
    """Send data to collection"""
    url = f"{BASE_URL}/projects/{project_id}/collections/{collection_id}/send_data"
    headers = {"X-API-Key": write_key}
    response = requests.post(url, json=data, headers=headers)
    if response.status_code == 200:
        print(f"✅ Sent {len(data)} records")
        return True
    else:
        print(f"Failed to send data: {response.text}")
        return False

# Generate and send data
soil_data = generate_soil_data(num_sensors=3, num_readings=24)
send_data(PROJECT_ID, collection_id, WRITE_KEY, soil_data)

✅ Sent 72 records


True

In [ ]:
def get_data(project_id, collection_id, read_key, filters=None, attributes = None, limit= None, order_by= None):
    """Get data from collection with filters"""
    url = f"{BASE_URL}/projects/{project_id}/collections/{collection_id}/get_data"
    headers = {"X-API-Key": read_key}
    params = {}
    if order_by:
      params["order_by"] = order_by
    if attributes:
      params["attributes"] = attributes
    if limit:
      params["limit"]= limit
    if filters:
        params["filters"] = json.dumps(filters)

    response = requests.get(url, headers=headers, params=params)
    if response.status_code == 200:
        data = response.json()
        return data
    else:
        print(f"Failed to get data: {response.text}")
        return []

# Get data
print("\n📥 Getting all data...")
all_data = get_data(PROJECT_ID, collection_id, READ_KEY)
print(all_data)
limit_data = get_data(PROJECT_ID, collection_id, READ_KEY, limit=14)
print(limit_data)
order_data = get_data(PROJECT_ID, collection_id, READ_KEY, attributes= ['key', 'temperature'], order_by='{"field": "temperature", "order": "desc"}')
print(order_data)


📥 Getting all data...
✅ Retrieved 72 records
[{'day': {'days_from_epoch': 20256}, 'key': 'SOIL_001', 'timestamp': '2025-06-17T08:04:23', 'battery': {'level': 81.80000305175781}, 'nitrogen': 47.400001525878906, 'ph': {'level': 7.099999904632568}, 'phosphorus': 10.199999809265137, 'potassium': 127.4000015258789, 'soil': {'moisture': 35.900001525878906}, 'temperature': 18.299999237060547}, {'day': {'days_from_epoch': 20256}, 'key': 'SOIL_001', 'timestamp': '2025-06-17T07:04:23', 'battery': {'level': 91.4000015258789}, 'nitrogen': 59.70000076293945, 'ph': {'level': 7.199999809265137}, 'phosphorus': 14.5, 'potassium': 130.39999389648438, 'soil': {'moisture': 39.099998474121094}, 'temperature': 23.5}, {'day': {'days_from_epoch': 20256}, 'key': 'SOIL_001', 'timestamp': '2025-06-17T06:04:23', 'battery': {'level': 80.4000015258789}, 'nitrogen': 31.100000381469727, 'ph': {'level': 6.099999904632568}, 'phosphorus': 16.600000381469727, 'potassium': 186.5, 'soil': {'moisture': 43.900001525878906},

In [ ]:
# Get data for specific sensor
print("\n🔍 Filter by sensor SOIL_001...")
sensor_filter = [
    {
        "property_name": "key",
        "operator": "eq",
        "property_value": "SOIL_001"
    }
]
sensor_data = get_data(PROJECT_ID, collection_id, READ_KEY, filters=sensor_filter)
print(sensor_data)
# Get high moisture readings
print("\n💧 Filter for high moisture (>45%)...")
moisture_filter = [
    {
        "property_name": "soil_moisture",
        "operator": "gt",
        "property_value": 45
    }
]
high_moisture = get_data(PROJECT_ID, collection_id, READ_KEY, moisture_filter)
print(high_moisture)
# Complex filter - low moisture AND low battery
print("\n⚠️ Filter for low moisture AND low battery...")
complex_filter = [
    {
        "property_name": "soil_moisture",
        "operator": "lt",
        "property_value": 35
    },
    {
        "property_name": "battery_level",
        "operator": "lt",
        "property_value": 80
    }
]
alert_data = get_data(PROJECT_ID, collection_id, READ_KEY, complex_filter)
print(alert_data)


🔍 Filter by sensor SOIL_001...
✅ Retrieved 24 records

💧 Filter for high moisture (>45%)...
✅ Retrieved 30 records

⚠️ Filter for low moisture AND low battery...
✅ Retrieved 2 records
[{'day': {'days_from_epoch': 20256}, 'key': 'SOIL_003', 'timestamp': '2025-06-17T03:04:23', 'battery': {'level': 76.0}, 'nitrogen': 53.5, 'ph': {'level': 7.0}, 'phosphorus': 10.600000381469727, 'potassium': 147.5, 'soil': {'moisture': 32.70000076293945}, 'temperature': 19.5}, {'day': {'days_from_epoch': 20255}, 'key': 'SOIL_002', 'timestamp': '2025-06-16T18:04:23', 'battery': {'level': 75.0}, 'nitrogen': 51.0, 'ph': {'level': 7.099999904632568}, 'phosphorus': 13.699999809265137, 'potassium': 150.3000030517578, 'soil': {'moisture': 31.899999618530273}, 'temperature': 23.899999618530273}]


In [ ]:
## Step 4: Get Statistics

def get_statistics(project_id, collection_id, read_key, attribute, stat="avg", filters=None, order="asc", interval="every_24_hours"):
    """Get statistics for attribute"""
    url = f"{BASE_URL}/projects/{project_id}/collections/{collection_id}/statistics"
    headers = {"X-API-Key": read_key}

    params = {
        "attribute": attribute,
        "stat": stat,
        "interval": interval,
        "order": order
    }

    if filters:
        params["filters"] = json.dumps(filters)

    response = requests.get(url, headers=headers, params=params)
    if response.status_code == 200:
        stats = response.json()
        print(f"✅ {stat} for {attribute}: {stats}")
        return stats
    else:
        print(f"Failed to get statistics: {response.text}")
        return {}

# Calculate statistics
print("\n📈 Getting statistics...")

# Average moisture
get_statistics(PROJECT_ID, collection_id, READ_KEY, "soil_moisture", "avg", )

# Maximum temperature
get_statistics(PROJECT_ID, collection_id, READ_KEY, "temperature", "max", order="desc")

# Minimum battery level
get_statistics(PROJECT_ID, collection_id, READ_KEY, "battery_level", "min", filters=[{"property_name": "key", "operator": "eq", "property_value": "SOIL_001"}])
# Distinct values of attribute
get_statistics(PROJECT_ID, collection_id, READ_KEY, "key", "distinct")


📈 Getting statistics...
✅ avg for soil_moisture: [{'key': 'SOIL_001', 'interval_start': '2025-06-16T09:00:00', 'avg_soil_moisture': 44.312}, {'key': 'SOIL_002', 'interval_start': '2025-06-16T09:00:00', 'avg_soil_moisture': 43.333}, {'key': 'SOIL_003', 'interval_start': '2025-06-16T09:00:00', 'avg_soil_moisture': 48.033}]
✅ max for temperature: [{'key': 'SOIL_001', 'interval_start': '2025-06-16T09:00:00', 'max_temperature': 24.6}, {'key': 'SOIL_002', 'interval_start': '2025-06-16T09:00:00', 'max_temperature': 24.9}, {'key': 'SOIL_003', 'interval_start': '2025-06-16T09:00:00', 'max_temperature': 25.0}]
✅ min for battery_level: [{'key': 'SOIL_001', 'interval_start': '2025-06-16T09:00:00', 'min_battery_level': 70.4}, {'key': 'SOIL_002', 'interval_start': '2025-06-16T09:00:00', 'min_battery_level': 70.6}, {'key': 'SOIL_003', 'interval_start': '2025-06-16T09:00:00', 'min_battery_level': 70.5}]


[{'key': 'SOIL_001',
  'interval_start': '2025-06-16T09:00:00',
  'min_battery_level': 70.4},
 {'key': 'SOIL_002',
  'interval_start': '2025-06-16T09:00:00',
  'min_battery_level': 70.6},
 {'key': 'SOIL_003',
  'interval_start': '2025-06-16T09:00:00',
  'min_battery_level': 70.5}]

In [ ]:
def delete_data(project_id, collection_id, master_key, key=None, timestamp_from=None, timestamp_to=None):
    """Delete data from collection based on criteria"""
    url = f"{BASE_URL}/projects/{project_id}/collections/{collection_id}/delete_data"
    headers = {"X-API-Key": master_key}

    delete_request = {}
    if key:
        delete_request["key"] = key
    if timestamp_from:
        delete_request["timestamp_from"] = timestamp_from
    if timestamp_to:
        delete_request["timestamp_to"] = timestamp_to

    response = requests.delete(url, json=delete_request, headers=headers)
    if response.status_code == 200:
        result = response.json()
        print(f"✅ Data deleted successfully: {result['message']}")
        return result
    else:
        print(f"Failed to delete data: {response.text}")
        return None
print(get_data(PROJECT_ID, collection_id, READ_KEY, attributes=['key', 'timestamp'],order_by='{"field": "timestamp", "order": "asc"}', filters=[
    {
        "property_name": "key",
        "operator": "eq",
        "property_value": "SOIL_001"
    }
]))
delete_data(PROJECT_ID, collection_id, MASTER_KEY, "SOIL_001", timestamp_from= '2025-09-18T00:00:00Z', timestamp_to= '2025-09-18T23:00:00Z')
print(get_data(PROJECT_ID, collection_id, READ_KEY, attributes=['key', 'timestamp'],order_by='{"field": "timestamp", "order": "asc"}', filters=[
    {
        "property_name": "key",
        "operator": "eq",
        "property_value": "SOIL_001"
    }
]))